# 02 - Two run comparison

This notebook compares two recordings:
- Time-domain overlays
- FFT overlays
- Dominant peak comparison


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from helpers import compute_fft, dominant_peaks, estimate_sampling_rate_hz, load_measurement_csv


## Select two files
Use any two runs to compare operating conditions.


In [ ]:
path_a = Path('../data/sample/sample_run_01.csv')
path_b = Path('../data/sample/sample_run_02.csv')

df_a = load_measurement_csv(str(path_a))
df_b = load_measurement_csv(str(path_b))

fs_a = estimate_sampling_rate_hz(df_a)
fs_b = estimate_sampling_rate_hz(df_b)
print(f'Run A sample rate: {fs_a:.2f} Hz')
print(f'Run B sample rate: {fs_b:.2f} Hz')


## Time-domain overlay
Compare raw acceleration trends axis by axis.


In [ ]:
t_a = df_a['timestamp_ms'] / 1000.0
t_b = df_b['timestamp_ms'] / 1000.0
fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for i, col in enumerate(['ax_g', 'ay_g', 'az_g']):
    ax[i].plot(t_a, df_a[col], label=f'Run A {col}')
    ax[i].plot(t_b, df_b[col], label=f'Run B {col}', linestyle='--')
    ax[i].grid(True, alpha=0.3)
    ax[i].legend(loc='upper right')
ax[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()


## Spectrum overlay
Compare dominant frequency content between runs.


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
all_rows = []
for i, col in enumerate(['ax_g', 'ay_g', 'az_g']):
    f_a, a_a = compute_fft(df_a[col].to_numpy(), fs_a)
    f_b, a_b = compute_fft(df_b[col].to_numpy(), fs_b)
    ax[i].plot(f_a, a_a, label=f'Run A {col}')
    ax[i].plot(f_b, a_b, label=f'Run B {col}', linestyle='--')
    ax[i].grid(True, alpha=0.3)
    ax[i].legend(loc='upper right')

    peaks_a = dominant_peaks(f_a, a_a, top_n=3).assign(run='A', axis=col)
    peaks_b = dominant_peaks(f_b, a_b, top_n=3).assign(run='B', axis=col)
    all_rows.extend([peaks_a, peaks_b])

ax[-1].set_xlabel('Frequency [Hz]')
plt.tight_layout()
plt.show()

peak_table = pd.concat(all_rows, ignore_index=True)
peak_table


## Interpretation notes
- Larger peak amplitude indicates stronger periodic vibration at that frequency.
- Peak shifts may indicate changing operating condition or mounting condition.
- Next step: correlate with RPM and test repeatability across multiple runs.
